# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()




,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# checking total number of duplicate rows
num_dupes = df.duplicated().sum()
print("Number of duplicates:", num_dupes)

#dropping the duplicate rows
df = df.drop_duplicates()

#logging the results
log('TODO 1', 'Dropped exact duplicate rows', num_dupes)


Number of duplicates: 15
[TODO 1] Dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
#stripping the $ character and converting it to float numbers
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).astype(float)

#logging step 2
log('TODO 2', 'Cleaned price column', 0)

(df['price'].astype(str).str.replace('$', '', regex=False).astype(float) * df['qty'].fillna(0)).sum()



[TODO 2] Cleaned price column (0 row(s))


np.float64(4594.5)

### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# converting 'qty' to numeric quantities
df['qty'] = pd.to_numeric(df['qty'])

#identifying missing or negative/0 quanitites
invalid_mask = df['qty'].isna() | (df['qty'] < 1)
invalid_count = invalid_mask.sum()
print("Invalid rows to drop:", invalid_count)

df = df[df['qty'] >= 1]

#logging step 3
log('TODO 3', 'Dropped missing and negative quantities', invalid_count)
(df['price'].astype(str).str.replace('$', '', regex=False).astype(float) * df['qty'].fillna(0)).sum()

Invalid rows to drop: 25
[TODO 3] Dropped missing and negative quantities (25 row(s))


np.float64(4740.0)

### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

#Item mapping based on results
ITEM_MAP = {'Cheeseburger': 'Cheeseburger',
            'cheese burger': 'Cheeseburger',
            'Foam Finger': 'Foam Finger',
            'foam finger': 'Foam Finger',
            'Foam Finter': 'Foam Finger',
            'Rain Poncho': 'Rain Poncho',
            'rain poncho': 'Rain Poncho',
}
#applying the mapping
df['item'] = df['item'].replace(ITEM_MAP)

#logging step 4
log('TODO 4', 'Canonicalized item names into 3 standard categories', len(df))


item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[TODO 4] Canonicalized item names into 3 standard categories (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())

#Category mapping based on results
#combined apparel with merch
CATEGORY_MAP = {'Food': 'Food',
            'food': 'Food',
            'Merch': 'Merch',
            'Apparel': 'Merch',
            'RainGear': 'Rain Gear',
            'rain-gear': 'Rain Gear',
}
df['category'] = df['category'].replace(CATEGORY_MAP)

#logging step 5
log('TODO 5', 'Canonicalized item names into 3 standard categories', len(df))

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[TODO 5] Canonicalized item names into 3 standard categories (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isin(['Foam Finger', 'Rain Poncho', 'Cheeseburger']).all()
assert df['category'].isin(['Food', 'Merch', 'Rain Gear']).all()
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
#calculation for revenue
df['revenue'] = df['price'] * df['qty']

#revenue by category, highest first
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
print("Revenue by Category (Highest First):")
print(revenue_by_category)

#overall total
total_revenue = df['revenue'].sum().round(2)
print("Overall Total Revenue:", total_revenue)


Revenue by Category (Highest First):
category
Food         1656.0
Merch        1572.0
Rain Gear    1512.0
Name: revenue, dtype: float64
Overall Total Revenue: 4740.0


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,TODO 1,Dropped exact duplicate rows,15
1,TODO 2,Cleaned price column,0
2,TODO 3,Dropped missing and negative quantities,25
3,TODO 4,Canonicalized item names into 3 standard categ...,275
4,TODO 5,Canonicalized item names into 3 standard categ...,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) The cleaning step that changed revenue the most was step 1, getting rid of the duplicates. This resulted in a $258 change, from $4,852.50 to $4594.50.

b) In TODO 5, I combined Merch and Apparel together, whereas a reasonable person might have combined Apparel and Rain Gear together. I just felt that rain gear should be left on its own while apparel and merch like the foam finger could be combined together to classify as a larger non-food merchandise category. It does not make any change to the reported total revenue but redistributes the revenue by category (merch revenue grows because it combines apparel while rain gear apparel doesn't include anything additional)